# 29. DETR 객체 탐지 Transformer

이 노트북은 `28_ViT_이미지_분류_실습.ipynb` 다음 단계로, DETR이 객체 탐지를 Transformer와 set prediction 문제로 바라보는 방식을 이해합니다.

기존 detection 모델은 anchor, proposal, NMS 같은 후처리 흐름이 중요했습니다. DETR은 object query를 사용해 정해진 개수의 예측 set을 만들고, Hungarian matching으로 ground truth와 일대일 매칭합니다.

이번 노트북의 목표는 다음과 같습니다.

- DETR의 큰 구조를 이해합니다.
- object query가 무엇을 의미하는지 파악합니다.
- set prediction과 no object class의 필요성을 이해합니다.
- Hungarian matching의 비용 계산을 작은 예제로 확인합니다.

## 29-1. 준비

실제 DETR 모델을 실행하지 않고, 작은 예측 box와 ground truth box로 matching 원리를 실습합니다. 외부 패키지 의존을 줄이기 위해 Hungarian matching은 작은 경우에 대해 brute force로 계산합니다.

In [ ]:
from itertools import permutations

import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)

## 29-2. DETR의 큰 흐름

DETR은 CNN backbone과 Transformer encoder-decoder를 함께 사용합니다.

```text
image
  -> CNN backbone feature map
  -> Transformer encoder
  -> Transformer decoder + object queries
  -> fixed-size set of predictions
  -> Hungarian matching during training
  -> class + box outputs
```

출력은 일정 개수의 prediction slot입니다. 실제 객체보다 slot이 많으므로, 객체가 없는 slot은 `no object` class가 됩니다.

## 29-3. Ground truth와 예측 set 만들기

DETR의 예측은 순서가 중요하지 않은 set입니다. 예측 0번이 반드시 첫 번째 객체를 의미하지 않습니다. 학습 중에는 각 예측 slot을 어떤 ground truth와 연결할지 matching으로 결정합니다.

In [ ]:
class_names = np.array(['person', 'car', 'dog', 'no object'])

# box format: x1, y1, x2, y2 in normalized coordinates
gt_boxes = np.array([
    [0.12, 0.18, 0.38, 0.72],
    [0.55, 0.42, 0.88, 0.70],
])
gt_labels = np.array([0, 1])

pred_boxes = np.array([
    [0.58, 0.40, 0.86, 0.69],
    [0.10, 0.20, 0.40, 0.70],
    [0.25, 0.08, 0.45, 0.30],
])
pred_prob = np.array([
    [0.08, 0.82, 0.04, 0.06],
    [0.76, 0.10, 0.06, 0.08],
    [0.10, 0.08, 0.12, 0.70],
])

print('ground truth objects:', len(gt_boxes))
print('prediction slots:', len(pred_boxes))
print('predicted classes:', class_names[pred_prob.argmax(axis=1)])

In [ ]:
def draw_boxes(ax, boxes, labels, color, title):
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor('#eef2f7')
    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box
        rect = Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5)
        ax.add_patch(rect)
        ax.text(x1, y1 - 0.02, label, color=color, weight='bold')

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
draw_boxes(axes[0], gt_boxes, class_names[gt_labels], '#16a34a', 'Ground truth')
draw_boxes(axes[1], pred_boxes, class_names[pred_prob.argmax(axis=1)], '#2563eb', 'Prediction set')
plt.show()

## 29-4. Matching 비용 계산

예측 slot과 ground truth를 연결할 때는 보통 class 비용과 box 비용을 함께 사용합니다. 여기서는 단순화를 위해 다음 비용을 사용합니다.

```text
cost = class_cost + L1_box_cost + (1 - IoU)
```

class_cost는 해당 ground truth class의 예측 확률이 높을수록 작아집니다.

In [ ]:
def box_iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0

num_pred = len(pred_boxes)
num_gt = len(gt_boxes)
cost = np.zeros((num_pred, num_gt))

for i in range(num_pred):
    for j in range(num_gt):
        class_cost = -pred_prob[i, gt_labels[j]]
        l1_cost = np.abs(pred_boxes[i] - gt_boxes[j]).sum()
        iou_cost = 1 - box_iou(pred_boxes[i], gt_boxes[j])
        cost[i, j] = class_cost + l1_cost + iou_cost

print('matching cost matrix: rows=prediction slots, cols=ground truth')
print(cost.round(3))

In [ ]:
best_total = float('inf')
best_assignment = None

for pred_indices in permutations(range(num_pred), num_gt):
    total = sum(cost[pred_idx, gt_idx] for gt_idx, pred_idx in enumerate(pred_indices))
    if total < best_total:
        best_total = total
        best_assignment = pred_indices

print('best assignment:')
for gt_idx, pred_idx in enumerate(best_assignment):
    print(f'  GT {gt_idx} ({class_names[gt_labels[gt_idx]]}) <- prediction slot {pred_idx}')
print('total cost:', round(best_total, 3))

매칭되지 않은 prediction slot은 `no object`로 학습됩니다. 이 구조 덕분에 DETR은 같은 객체에 대한 중복 box를 많이 만들고 NMS로 지우는 방식과 다르게, 학습 단계에서 일대일 예측을 강하게 유도합니다.

In [ ]:
matched_pred = set(best_assignment)
for i in range(num_pred):
    if i not in matched_pred:
        print(f'prediction slot {i} is trained as no object')

## 29-5. DETR이 기존 detector와 다른 점

| 관점 | YOLO/Faster R-CNN 계열 | DETR |
|---|---|---|
| 예측 방식 | dense prediction 또는 proposal 기반 | fixed-size set prediction |
| 중복 제거 | NMS가 중요 | 일대일 matching으로 중복 억제 |
| 핵심 구성 | anchor, grid, proposal 등 | object query, Transformer decoder |
| 학습 매칭 | anchor/positive assignment | Hungarian matching |

DETR은 구조가 단순하고 end-to-end 관점이 강하지만, 초기 DETR은 학습 수렴이 느린 편이었습니다. 이후 Deformable DETR 등은 attention을 더 효율적으로 바꿔 이 문제를 개선했습니다.

## 정리

- DETR은 객체 탐지를 fixed-size set prediction 문제로 봅니다.
- object query는 decoder에서 각 예측 slot의 역할을 맡습니다.
- Hungarian matching은 prediction slot과 ground truth를 일대일로 연결합니다.
- 매칭되지 않은 slot은 `no object` class로 학습됩니다.

다음 노트북 `30_Swin_Transformer_계층적_비전_모델.ipynb`에서는 Transformer를 비전 backbone으로 쓰기 위한 window attention과 계층 구조를 살펴봅니다.